In [1]:
# 1. Imports and setup
import json
import re
import emoji
import asyncio
from unidecode import unidecode
from googletrans import Translator
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk
# 1. Back translation
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

nltk.data.path.append(r'C:/Users/Marvel Wilbert O/AppData/Roaming/nltk_data')
# nltk.download('stopwords', download_dir=r'C:/Users/Marvel Wilbert O/AppData/Roaming/nltk_data')
# nltk.download('popular', download_dir=r'C:/Users/Marvel Wilbert O/AppData/Roaming/nltk_data')


In [2]:

def remove_extra_chars(text):
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    return text

def convert_emojis(text):
    return emoji.demojize(text, delimiters=(" ", " "), language='id')

def remove_usernames(text):
    return re.sub(r'@\w+', '@USER', text)

def remove_numbers(text):
    return re.sub(r'\d+', '', text)

def remove_punctuation(text):
    return re.sub(r'[^\w\s{}]', ' ', text)

def replace_links(text):
    return re.sub(r'http[s]?://\S+|www\.\S+', 'HTTPURL', text)

def normalize_text(text):
    # Convert to ASCII, remove unsupported formatting
    return unidecode(str(text))

def lowercase(text):
    return text.lower()

def normalize_whitespace(text):
    return re.sub(r'\s+', ' ', text).strip()


In [5]:
# 1. Back translation
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

async def test_preprocessing_order(sample_text):

    text = convert_emojis(sample_text)
    print("Convert emoji:", text)

    text = normalize_text(text)
    print("Normalize (unidecode):", text)

    text = remove_extra_chars(text)
    print("Remove extra chars:", text)

    # 6. Lowercase
    text = text.lower()
    print("Lowercase:", text)

    # 8. Replace URL
    text = replace_links(text)
    print("Replace URL:", text)

    # 9. Remove numbers
    text = remove_numbers(text)
    print("Remove numbers:", text)

    # 10. Remove punctuation
    text = remove_punctuation(text)
    print("Remove punctuation:", text)

    text = normalize_whitespace(text)
    print("Normalize whitespace:", text)


# Example usage:
sample_text = "Halo gaes!!! Cek link ini ya: https://example.com 😊😊😊 #excited @friend123 12345"
await test_preprocessing_order(sample_text)

Convert emoji: Halo gaes!!! Cek link ini ya: https://example.com  wajah_tersenyum_dengan_mata_bahagia  wajah_tersenyum_dengan_mata_bahagia  wajah_tersenyum_dengan_mata_bahagia  #excited @friend123 12345
Normalize (unidecode): Halo gaes!!! Cek link ini ya: https://example.com  wajah_tersenyum_dengan_mata_bahagia  wajah_tersenyum_dengan_mata_bahagia  wajah_tersenyum_dengan_mata_bahagia  #excited @friend123 12345
Remove extra chars: Halo gaes! Cek link ini ya: https://example.com  wajah_tersenyum_dengan_mata_bahagia  wajah_tersenyum_dengan_mata_bahagia  wajah_tersenyum_dengan_mata_bahagia  #excited @friend123 12345
Lowercase: halo gaes! cek link ini ya: https://example.com  wajah_tersenyum_dengan_mata_bahagia  wajah_tersenyum_dengan_mata_bahagia  wajah_tersenyum_dengan_mata_bahagia  #excited @friend123 12345
Replace URL: halo gaes! cek link ini ya: HTTPURL  wajah_tersenyum_dengan_mata_bahagia  wajah_tersenyum_dengan_mata_bahagia  wajah_tersenyum_dengan_mata_bahagia  #excited @friend123 12

In [ ]:
import json

async def run_preprocessing_steps_separate_files(start_idx=0, checkpoint_interval=5500):

    with open('fetched_data.json', 'r', encoding='utf-8') as f:
        data = json.load(f)

    final_results = []

    if start_idx > 0:
        try:
            with open(f'result_BERT/fetched_data_final_checkpoint_{start_idx}.json', 'r', encoding='utf-8') as f:
                final_results = json.load(f)
            print(f"Loaded checkpoint at item {start_idx}")
        except FileNotFoundError:
            print(f"Checkpoint files for index {start_idx} not found. Starting from scratch.")
            start_idx = 0

    for idx, item in enumerate(data[start_idx:], start=start_idx):
        text = item['text']
        label = item.get('votes', None)  

        text = convert_emojis(text)

        text = normalize_text(text)

        text = remove_extra_chars(text)
        
        text = text.lower()
        
        text = replace_links(text)
        
        text = remove_numbers(text)
        
        text = remove_punctuation(text)
        
        text = normalize_whitespace(text)
        
        final_results.append({
            'text': text,
            'label': label
        })

        # Checkpoint: save every 100 items
        if (idx + 1) % checkpoint_interval == 0:
            checkpoint_num = idx + 1
            with open(f'result_BERT/fetched_data_final_checkpoint_{checkpoint_num}.json', 'w', encoding='utf-8') as f:
                json.dump(final_results, f, ensure_ascii=False, indent=2)
            print(f'Checkpoint saved at item {checkpoint_num}')

    # Save final results
    with open('result_BERT/fetched_data_final.json', 'w', encoding='utf-8') as f:
        json.dump(final_results, f, ensure_ascii=False, indent=2)
    print('Saved final results to result_BERT/fetched_data_final.json')

await run_preprocessing_steps_separate_files()

Checkpoint saved at item 5500
Checkpoint saved at item 11000
Saved final results to result_BERT/fetched_data_final.json


In [2]:
import json
from collections import defaultdict

with open('result_BERT/fetched_data_final.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

text_to_labels = defaultdict(set)
for item in data:
    text = item['text']
    label = item['label']
    if text is not None and label is not None:
        text_to_labels[text].add(label)

conflicting = {text: labels for text, labels in text_to_labels.items() if len(labels) > 1}
print(f"Number of duplicate texts with conflicting labels: {len(conflicting)}")
if conflicting:
    print("Examples of conflicts:")
    for text, labels in list(conflicting.items())[:5]:
        print(f"Text: {text}\nLabels: {labels}\n")

Number of duplicate texts with conflicting labels: 9
Examples of conflicts:
Text: back fight adalah aplikasi penghasil uang tercepat sejauh ini jika kamu tidak datang kamu akan rugi masukkan id dan dapatkan rpk segera HTTPURL
Labels: {False, True}

Text: back fight adalah aplikasi penghasil uang yang memungkinkan anda menerima dana dengan sangat cepat anda belum menginstalnya cukup masukkan id undangan untuk menerima rpk HTTPURL
Labels: {False, True}

Text: jika anda ingin menghasilkan uang instal speed man semakin awal anda datang semakin banyak yang bisa anda hasilkan masukkan id ini untuk menerima manfaat rpk HTTPURL
Labels: {False, True}

Text: jika anda ingin menghasilkan uang instal back fight semakin awal anda datang semakin banyak yang dapat anda peroleh masukkan id ini untuk menerima manfaat rpk HTTPURL
Labels: {False, True}

Text: aplikasi baru back fight ini menghasilkan uang dengan sangat cepat masukkan id undangan dan anda bisa mendapatkan rpk HTTPURL
Labels: {False, True}

In [3]:
import json
from collections import defaultdict, Counter

# Load data
with open('result_BERT/fetched_data_final.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# Group labels for each text
text_to_labels = defaultdict(list)
for item in data:
    text = item['text']
    label = item['label']
    if text is not None and label is not None:
        text_to_labels[text].append(label)

# Build unique data with majority vote for label
unique_data = []
for text, labels in text_to_labels.items():
    label_counts = Counter(labels)
    majority_label = label_counts.most_common(1)[0][0]
    unique_data.append({'text': text, 'label': majority_label})

print(f"Original data size: {len(data)}")
print(f"Unique data size: {len(unique_data)}")

# Optionally, save the deduplicated data
with open('result_BERT/fetched_data_final_dedup.json', 'w', encoding='utf-8') as f:
    json.dump(unique_data, f, ensure_ascii=False, indent=2)
print('Saved deduplicated data to result_BERT/fetched_data_final_dedup.json')

Original data size: 12058
Unique data size: 11693
Saved deduplicated data to result_BERT/fetched_data_final_dedup.json
